In [ ]:
import json
from collections import Counter

file_path = "data/techmap_jobs_sample_30000.jsonl"

field_counter = Counter()
sample_records = []

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 1000:
            break

        record = json.loads(line)
        field_counter.update(record.keys())

        if i < 3:
            sample_records.append(record)

print("Records inspected:", i + 1)

print("\nMost common fields:")
for field, count in field_counter.most_common():
    print(field, count)

print("\nSample record keys:")
print(sample_records[0].keys())

print("\nFirst sample record:")
sample = sample_records[0]
for k, v in sample.items():
    print(f"{k}: {str(v)[:300]}")

In [ ]:
total = sum(weights.values())

weights = {
    k: v/total
    for k,v in weights.items()
}

weights

In [ ]:
import sys
!{sys.executable} -m pip install pandas tqdm

In [ ]:
import pandas as pd
from tqdm import tqdm

print("pandas works")

In [ ]:
import json
import pandas as pd
import hashlib
from tqdm import tqdm

file_path = "data/techmap_jobs_sample_30000.jsonl"
output_path = "data/jobpilot_jobs_snapshot.csv"

MAX_RECORDS = 25000

def safe_get_date(value):
    if isinstance(value, dict) and "$date" in value:
        return value["$date"]
    return value

def extract_company(value):
    if isinstance(value, dict):
        return (
            value.get("name")
            or value.get("companyName")
            or value.get("title")
            or value.get("idInSource")
            or ""
        )
    return value or ""

def extract_location(value):
    if isinstance(value, dict):
        return (
            value.get("addressLine")
            or value.get("city")
            or value.get("region")
            or value.get("country")
            or ""
        )
    return value or ""

def make_hash(title, company, location, description):
    raw = f"{title}|{company}|{location}|{description[:500]}".lower().strip()
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

rows = []
seen_hashes = set()

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in tqdm(f, total=MAX_RECORDS):
        if len(rows) >= MAX_RECORDS:
            break

        try:
            record = json.loads(line)
        except:
            continue

        title = record.get("name", "")
        company = extract_company(record.get("orgCompany", ""))
        location = extract_location(record.get("orgAddress", ""))
        description = record.get("text", "")
        job_url = record.get("url", "")
        country = record.get("sourceCC", "")
        date_posted = safe_get_date(record.get("dateCreated", ""))

        if not title or not description:
            continue

        if not company:
            company = "Unknown Company"

        dedup_key = make_hash(title, company, location, description)

        if dedup_key in seen_hashes:
            continue

        seen_hashes.add(dedup_key)

        rows.append({
            "job_id": len(rows) + 1,
            "title": title,
            "company": company,
            "location": location,
            "country": country,
            "description": description,
            "job_url": job_url,
            "date_posted": date_posted,
            "salary_min": None,
            "salary_max": None,
            "salary_text": "",
            "source": "Techmap Kaggle",
            "dedup_key": dedup_key
        })

jobs_df = pd.DataFrame(rows)

print("Clean records:", len(jobs_df))
print(jobs_df[["title", "company", "location", "country"]].head(10))
print(jobs_df.info())

jobs_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

In [ ]:
import pandas as pd

snapshot_path = "data/jobpilot_jobs_snapshot.csv"

jobs_df = pd.read_csv(snapshot_path)

print("Rows:", len(jobs_df))
print("Unique dedup keys:", jobs_df["dedup_key"].nunique())
print("Duplicate dedup keys:", jobs_df["dedup_key"].duplicated().sum())

print("\nMissing values:")
print(jobs_df[["title", "company", "location", "description", "job_url"]].isna().sum())

print("\nTop companies:")
print(jobs_df["company"].value_counts().head(10))

print("\nTop locations:")
print(jobs_df["location"].value_counts().head(10))

In [ ]:
import sys
!{sys.executable} -m pip install pypdf

In [ ]:
import pypdf
print("pypdf works")

In [ ]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """
    Extract all text from a PDF resume.
    """

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [ ]:
resume_path = "data/sample_resume.pdf"

resume_text = extract_text_from_pdf(resume_path)

print(resume_text[:3000])

In [ ]:
import re

SKILL_KEYWORDS = [
    # Programming / analytics
    "python", "r", "sql", "pandas", "numpy", "scikit-learn", "sklearn",
    "matplotlib", "seaborn", "tableau", "power bi", "excel",

    # Data engineering / big data
    "pyspark", "spark", "databricks", "mongodb", "mysql", "sqlite",
    "etl", "api", "data pipeline", "data cleaning",

    # Machine learning / statistics
    "machine learning", "logistic regression", "regression", "classification",
    "clustering", "pca", "random forest", "decision tree", "nlp",
    "sentiment analysis", "topic modeling",

    # Business / operations
    "data analytics", "business analytics", "dashboard", "visualization",
    "quality engineering", "process improvement", "cross-functional",
    "root cause analysis", "documentation",

    # Tools / engineering
    "solidworks", "catia", "jira", "git", "github"
]

def extract_skills(text, skill_keywords=SKILL_KEYWORDS):
    """
    Extract skills from resume text using a controlled keyword dictionary.
    """
    text_lower = text.lower()
    matched_skills = []

    for skill in skill_keywords:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, text_lower):
            matched_skills.append(skill)

    return sorted(set(matched_skills))

resume_skills = extract_skills(resume_text)

print("Skills found:")
print(resume_skills)
print("\nNumber of skills found:", len(resume_skills))

In [ ]:
def build_user_profile(
    resume_text,
    skills,
    target_roles=None,
    preferred_locations=None,
    salary_min=None,
    dealbreakers=None
):
    """
    Build a standardized user profile object for retrieval and ranking.
    """

    if target_roles is None:
        target_roles = ["Data Analyst", "Business Analyst", "Analytics Engineer", "Data Scientist"]

    if preferred_locations is None:
        preferred_locations = ["Remote", "Bay Area", "San Francisco", "Oakland", "San Jose"]

    if dealbreakers is None:
        dealbreakers = ["contract only", "unpaid", "senior", "staff", "principal"]

    profile = {
        "resume_text": resume_text,
        "skills": skills,
        "target_roles": target_roles,
        "preferred_locations": preferred_locations,
        "salary_min": salary_min,
        "dealbreakers": dealbreakers
    }

    return profile

user_profile = build_user_profile(
    resume_text=resume_text,
    skills=resume_skills,
    salary_min=80000
)

user_profile

In [ ]:
import re

def infer_career_level(resume_text):

    text = resume_text.lower()

    senior_terms = [
        "senior",
        "sr.",
        "staff",
        "principal",
        "lead",
        "manager",
        "director",
        "vice president",
        "vp"
    ]

    junior_terms = [
        "intern",
        "internship",
        "student",
        "graduate",
        "junior",
        "jr."
    ]

    # Strong senior signal
    for term in senior_terms:
        if term in text:
            return "senior"

    # Strong junior signal
    for term in junior_terms:
        if term in text:
            return "early"

    # Estimate based on years appearing in resume
    years = re.findall(r"(20\d{2})", resume_text)

    if len(years) >= 2:
        try:
            years = sorted([int(y) for y in years])

            experience_years = years[-1] - years[0]

            if experience_years <= 2:
                return "early"
            elif experience_years <= 7:
                return "mid"
            else:
                return "senior"

        except:
            pass

    return "early"

In [ ]:
user_profile = build_user_profile(
    resume_text=resume_text,
    skills=resume_skills,
    salary_min=80000
)

user_profile["career_level"] = infer_career_level(
    resume_text
)

print("Career Level:", user_profile["career_level"])

user_profile

In [ ]:
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

print("SentenceTransformers and FAISS work")

In [ ]:
import sys

!{sys.executable} -m pip install "numpy<2" --force-reinstall

In [ ]:
import numpy as np
print(np.__version__)

from sentence_transformers import SentenceTransformer
import faiss

print("SentenceTransformers and FAISS work")

In [ ]:
jobs_df["embedding_text"] = (
    "Title: " + jobs_df["title"].fillna("") + ". " +
    "Company: " + jobs_df["company"].fillna("") + ". " +
    "Location: " + jobs_df["location"].fillna("") + ". " +
    "Description: " + jobs_df["description"].fillna("")
)

print(jobs_df["embedding_text"].iloc[0][:1000])

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

job_texts = jobs_df["embedding_text"].tolist()

job_embeddings = model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", job_embeddings.shape)

In [ ]:
#Build FAISS Index

import faiss

embedding_dim = job_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(job_embeddings)

print("Number of vectors in FAISS index:", index.ntotal)

In [ ]:
#create user profile text

def build_profile_text(user_profile):
    profile_text = ""

    profile_text += "Target roles: " + ", ".join(user_profile["target_roles"]) + ". "
    profile_text += "Skills: " + ", ".join(user_profile["skills"]) + ". "
    profile_text += "Preferred locations: " + ", ".join(user_profile["preferred_locations"]) + ". "
    profile_text += "Resume summary: " + user_profile["resume_text"][:3000]

    return profile_text

profile_text = build_profile_text(user_profile)

print(profile_text[:1000])

In [ ]:
#retrieve top jobs

def retrieve_jobs(profile_text, model, index, jobs_df, top_k=100):
    profile_embedding = model.encode(
        [profile_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(profile_embedding, top_k)

    results = jobs_df.iloc[indices[0]].copy()
    results["semantic_similarity"] = scores[0]

    return results.reset_index(drop=True)

retrieved_jobs = retrieve_jobs(
    profile_text=profile_text,
    model=model,
    index=index,
    jobs_df=jobs_df,
    top_k=100
)

retrieved_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "job_url"
]].head(10)

In [ ]:
#save embeddings and faiss to prevent waiting again if kernel restarts in future

import numpy as np
import faiss

np.save(
    "artifacts/job_embeddings.npy",
    job_embeddings
)

faiss.write_index(
    index,
    "artifacts/job_faiss.index"
)

jobs_df.to_csv(
    "data/jobpilot_jobs_with_embedding_text.csv",
    index=False
)

print("Saved embeddings, FAISS index, and jobs dataframe.")

In [ ]:
#run retrieval 

retrieved_jobs = retrieve_jobs(
    profile_text=profile_text,
    model=model,
    index=index,
    jobs_df=jobs_df,
    top_k=100
)

retrieved_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "job_url"
]].head(10)

In [ ]:
#Apply hard filter

def apply_hard_filters(candidates_df, user_profile):
    """
    Remove jobs that violate seniority or dealbreaker constraints
    based on the user's career level.
    """

    filtered = candidates_df.copy()

    career_level = user_profile.get("career_level", "early").lower()

    senior_terms = [
        "senior", "sr.", "sr ", "staff", "principal",
        "lead", "manager", "director", "head of", "vp"
    ]

    junior_terms = [
        "junior", "jr.", "jr ", "entry level",
        "graduate", "intern", "internship"
    ]

    contract_terms = [
    "contract only",
    "unpaid",
    "temporary position",
    "temporary role",
    "contract position",
    "contract role"
]

    def violates_filter(row):
        title = str(row.get("title", "")).lower()
        description = str(row.get("description", "")).lower()

        # Always remove unpaid/temp/contract-only roles if present
        combined_text = title + " " + description
        for term in contract_terms:
            if term in combined_text:
                return True

        # Early-career candidates should not receive senior roles
        if career_level in ["student", "early", "new_grad"]:
            for term in senior_terms:
                if term in title:
                    return True

        # Senior candidates should not receive junior roles
        if career_level in ["senior", "experienced"]:
            for term in junior_terms:
                if term in title:
                    return True

        return False

    filtered["violates_dealbreaker"] = filtered.apply(violates_filter, axis=1)

    removed_jobs = filtered[filtered["violates_dealbreaker"] == True]
    kept_jobs = filtered[filtered["violates_dealbreaker"] == False].copy()

    return kept_jobs.reset_index(drop=True), removed_jobs.reset_index(drop=True)

In [ ]:
filtered_jobs, removed_jobs = apply_hard_filters(
    retrieved_jobs,
    user_profile
)

print("Career level:", user_profile["career_level"])
print("Retrieved jobs:", len(retrieved_jobs))
print("Kept after hard filters:", len(filtered_jobs))
print("Removed by hard filters:", len(removed_jobs))

removed_jobs[["title", "company"]].head(20)

In [ ]:
import re
import numpy as np

def normalize_score(series):
    min_val = series.min()
    max_val = series.max()

    if max_val == min_val:
        return np.ones(len(series))

    return (series - min_val) / (max_val - min_val)


def compute_skill_match(job_text, user_skills):
    job_text = str(job_text).lower()
    matched = []

    for skill in user_skills:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"
        if re.search(pattern, job_text):
            matched.append(skill)

    if len(user_skills) == 0:
        return 0, matched

    return len(matched) / len(user_skills), matched


def compute_role_match(title, target_roles):
    title = str(title).lower()

    for role in target_roles:
        role_lower = role.lower()

        if role_lower in title:
            return 1.0

        role_words = role_lower.split()
        if all(word in title for word in role_words):
            return 1.0

    return 0.0


def compute_location_match(location, preferred_locations):
    location = str(location).lower()

    for preferred in preferred_locations:
        preferred = preferred.lower()

        if preferred in location:
            return 1.0

        if preferred == "bay area" and any(
            city in location for city in ["san francisco", "oakland", "san jose", "berkeley", "palo alto"]
        ):
            return 1.0

        if preferred == "california" and any(
            ca in location for ca in ["ca", "california", "san francisco", "san jose", "oakland", "los angeles", "san diego"]
        ):
            return 1.0

    return 0.0


def rank_jobs(filtered_jobs, user_profile):
    ranked = filtered_jobs.copy()

    ranked["combined_job_text"] = (
        ranked["title"].fillna("") + " " +
        ranked["description"].fillna("")
    )

    skill_results = ranked["combined_job_text"].apply(
        lambda text: compute_skill_match(text, user_profile["skills"])
    )

    ranked["skill_match_score"] = skill_results.apply(lambda x: x[0])
    ranked["matched_skills"] = skill_results.apply(lambda x: x[1])

    ranked["role_match_score"] = ranked["title"].apply(
        lambda title: compute_role_match(title, user_profile["target_roles"])
    )

    ranked["location_match_score"] = ranked["location"].apply(
        lambda location: compute_location_match(location, user_profile["preferred_locations"])
    )

    ranked["semantic_score_norm"] = normalize_score(ranked["semantic_similarity"])

    ranked["final_score"] = (
        0.50 * ranked["semantic_score_norm"] +
        0.25 * ranked["skill_match_score"] +
        0.15 * ranked["role_match_score"] +
        0.10 * ranked["location_match_score"]
    )

    ranked = ranked.sort_values("final_score", ascending=False)

    return ranked.reset_index(drop=True)


ranked_jobs = rank_jobs(filtered_jobs, user_profile)

ranked_jobs[[
    "job_id",
    "title",
    "company",
    "location",
    "semantic_similarity",
    "skill_match_score",
    "role_match_score",
    "location_match_score",
    "final_score",
    "matched_skills"
]].head(10)

In [ ]:
def generate_explanation(job_row, user_profile):

    explanation_parts = []

    # Skills
    matched_skills = job_row["matched_skills"]

    if len(matched_skills) > 0:
        explanation_parts.append(
            f"Matched skills: {', '.join(matched_skills)}"
        )

    # Role alignment
    if job_row["role_match_score"] > 0:
        explanation_parts.append(
            "Matches one of your target roles."
        )

    # Location alignment
    if job_row["location_match_score"] > 0:
        explanation_parts.append(
            "Matches one of your preferred locations."
        )

    # Semantic similarity
    explanation_parts.append(
        f"Semantic similarity score: {job_row['semantic_similarity']:.3f}"
    )

    explanation_parts.append(
        f"Final recommendation score: {job_row['final_score']:.3f}"
    )

    return " | ".join(explanation_parts)

In [ ]:
ranked_jobs["explanation"] = ranked_jobs.apply(
    lambda row: generate_explanation(
        row,
        user_profile
    ),
    axis=1
)

ranked_jobs[[
    "title",
    "company",
    "final_score",
    "explanation"
]].head(10)

In [ ]:
#Adaptive Learning Feedback

ranking_weights = {
    "semantic": 0.50,
    "skill": 0.25,
    "role": 0.15,
    "location": 0.10
}

def rerank_with_weights(filtered_jobs, user_profile, weights):
    ranked = rank_jobs(filtered_jobs, user_profile).copy()

    ranked["final_score"] = (
        weights["semantic"] * ranked["semantic_score_norm"] +
        weights["skill"] * ranked["skill_match_score"] +
        weights["role"] * ranked["role_match_score"] +
        weights["location"] * ranked["location_match_score"]
    )

    ranked = ranked.sort_values("final_score", ascending=False).reset_index(drop=True)

    ranked["explanation"] = ranked.apply(
        lambda row: generate_explanation(row, user_profile),
        axis=1
    )

    return ranked


def update_weights_from_feedback(weights, feedback_rows):
    """
    feedback_rows should include:
    - feedback: accept, reject, or skip
    - skill_match_score
    - role_match_score
    - location_match_score
    - semantic_score_norm
    """

    new_weights = weights.copy()

    accepted = feedback_rows[feedback_rows["feedback"] == "accept"]
    rejected = feedback_rows[feedback_rows["feedback"] == "reject"]

    if len(accepted) > 0:
        if accepted["skill_match_score"].mean() > feedback_rows["skill_match_score"].mean():
            new_weights["skill"] += 0.05

        if accepted["location_match_score"].mean() > feedback_rows["location_match_score"].mean():
            new_weights["location"] += 0.05

        if accepted["role_match_score"].mean() > feedback_rows["role_match_score"].mean():
            new_weights["role"] += 0.05

    if len(rejected) > 0:
        if rejected["location_match_score"].mean() < feedback_rows["location_match_score"].mean():
            new_weights["location"] += 0.03

        if rejected["skill_match_score"].mean() < feedback_rows["skill_match_score"].mean():
            new_weights["skill"] += 0.03

    total = sum(new_weights.values())

    for key in new_weights:
        new_weights[key] = new_weights[key] / total

    return new_weights

In [ ]:
# Simulated feedback on current top 10
feedback_sample = ranked_jobs.head(10).copy()

feedback_sample["feedback"] = [
    "accept",  # Data Analytics Engineer Remote
    "accept",  # Data Scientist Intermediate
    "accept",  # Junior Data Scientist
    "skip",
    "accept",
    "accept",
    "reject",
    "skip",
    "reject",
    "accept"
]

updated_weights = update_weights_from_feedback(
    ranking_weights,
    feedback_sample
)

print("Original weights:")
print(ranking_weights)

print("\nUpdated weights:")
print(updated_weights)

adaptive_ranked_jobs = rerank_with_weights(
    filtered_jobs,
    user_profile,
    updated_weights
)

adaptive_ranked_jobs[[
    "title",
    "company",
    "location",
    "final_score",
    "matched_skills",
    "explanation"
]].head(10)

In [ ]:
def evaluate_top_k(ranked_df, k=10):
    top_k = ranked_df.head(k)

    avg_final_score = top_k["final_score"].mean()
    avg_skill_score = top_k["skill_match_score"].mean()
    avg_role_score = top_k["role_match_score"].mean()
    avg_location_score = top_k["location_match_score"].mean()

    return {
        "avg_final_score": avg_final_score,
        "avg_skill_score": avg_skill_score,
        "avg_role_score": avg_role_score,
        "avg_location_score": avg_location_score
    }


before_metrics = evaluate_top_k(ranked_jobs, k=10)
after_metrics = evaluate_top_k(adaptive_ranked_jobs, k=10)

benchmark_df = pd.DataFrame([
    {"stage": "Before feedback", **before_metrics},
    {"stage": "After feedback", **after_metrics}
])

benchmark_df

In [ ]:
import sys
!{sys.executable} -m pip install -U google-genai

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("file.env")

print("Gemini key loaded:", os.getenv("GEMINI_API_KEY") is not None)

In [ ]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Write one sentence confirming that the JobPilot resume generator is connected."
)

print(response.text)

In [ ]:
import time

def generate_tailored_resume(
    resume_text,
    selected_job,
    client,
    model_name="gemini-2.5-flash",
    max_retries=3
):
    prompt = f"""
You are an expert resume writer.

Your task is to tailor the candidate's resume to the job below.

IMPORTANT RULES:
- Do NOT invent employers.
- Do NOT invent degrees.
- Do NOT invent dates.
- Do NOT invent projects.
- Only rewrite and emphasize existing experience.
- Improve keyword alignment with the target job.
- Highlight relevant skills from the original resume.
- Keep the resume professional and ATS-friendly.

TARGET JOB:
Title: {selected_job['title']}
Company: {selected_job['company']}
Location: {selected_job['location']}
Job Description: {selected_job['description'][:4000]}

ORIGINAL RESUME:
{resume_text[:6000]}

OUTPUT FORMAT:
1. Professional Summary
2. Key Skills
3. Tailored Experience Bullets
4. Why This Resume Fits The Job
"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            return response.text

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))

    return "Resume generation failed because the model was temporarily unavailable. Please try again."

In [ ]:
selected_job = ranked_jobs.iloc[0]

print(selected_job["title"])
print(selected_job["company"])

In [ ]:
tailored_resume = generate_tailored_resume(
    resume_text=resume_text,
    selected_job=selected_job,
    client=client
)

print(tailored_resume)

In [ ]:
#Batch analytics

from collections import Counter
import pandas as pd
import re

ANALYTICS_SKILLS = [
    "python", "sql", "r", "excel", "tableau", "power bi",
    "databricks", "spark", "pyspark", "aws", "azure", "gcp",
    "machine learning", "nlp", "regression", "clustering",
    "pandas", "numpy", "scikit-learn", "etl", "api",
    "dashboard", "visualization"
]

def count_skill_mentions(jobs_df, skills=ANALYTICS_SKILLS):
    counts = Counter()

    for text in jobs_df["description"].fillna("").astype(str):
        text_lower = text.lower()

        for skill in skills:
            pattern = r"\b" + re.escape(skill.lower()) + r"\b"
            if re.search(pattern, text_lower):
                counts[skill] += 1

    return pd.DataFrame(
        counts.items(),
        columns=["skill", "job_count"]
    ).sort_values("job_count", ascending=False)


top_skills_df = count_skill_mentions(jobs_df)

top_locations_df = (
    jobs_df["location"]
    .value_counts()
    .reset_index()
    .rename(columns={"location": "job_count", "index": "location"})
)

top_companies_df = (
    jobs_df["company"]
    .value_counts()
    .reset_index()
    .rename(columns={"company": "job_count", "index": "company"})
)

top_titles_df = (
    jobs_df["title"]
    .value_counts()
    .reset_index()
    .rename(columns={"title": "job_count", "index": "title"})
)

salary_summary = {
    "total_jobs": len(jobs_df),
    "jobs_with_salary_min": jobs_df["salary_min"].notna().sum(),
    "jobs_with_salary_max": jobs_df["salary_max"].notna().sum(),
    "jobs_with_salary_text": jobs_df["salary_text"].fillna("").astype(str).str.strip().ne("").sum()
}

print("Top Skills")
display(top_skills_df.head(15))

print("Top Locations")
display(top_locations_df.head(15))

print("Top Companies")
display(top_companies_df.head(15))

print("Top Titles")
display(top_titles_df.head(15))

print("Salary Summary")
salary_summary

In [ ]:
#Adaptive Learning from Feedback

adaptive_df = ranked_jobs.copy()

print(adaptive_df.shape)
adaptive_df.head()

In [ ]:
def simulate_feedback(row):

    if (
        row["skill_match_score"] >= 0.25 and
        row["role_match_score"] >= 1.0
    ):
        return "accept"

    elif (
        row["skill_match_score"] <= 0.10 and
        row["role_match_score"] == 0
    ):
        return "reject"

    else:
        return "skip"

In [ ]:
adaptive_df["feedback"] = adaptive_df.apply(
    simulate_feedback,
    axis=1
)

adaptive_df["feedback"].value_counts()

In [ ]:
adaptive_df["original_rank"] = (
    np.arange(len(adaptive_df)) + 1
)

accepted_jobs = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

baseline_rank = accepted_jobs["original_rank"].mean()

print("Baseline average accepted-job rank:")
print(round(baseline_rank,2))

In [ ]:
weights = {
    "semantic": 0.50,
    "skill": 0.25,
    "role": 0.15,
    "location": 0.10
}

weights

In [ ]:
accepted = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

rejected = adaptive_df[
    adaptive_df["feedback"] == "reject"
]

In [ ]:
skill_diff = (
    accepted["skill_match_score"].mean()
    - rejected["skill_match_score"].mean()
)

role_diff = (
    accepted["role_match_score"].mean()
    - rejected["role_match_score"].mean()
)

location_diff = (
    accepted["location_match_score"].mean()
    - rejected["location_match_score"].mean()
)

In [ ]:
weights["skill"] += 0.50 * skill_diff
weights["role"] += 0.50 * role_diff
weights["location"] += 0.50 * location_diff

In [ ]:
total = sum(weights.values())

weights = {
    k: v/total
    for k,v in weights.items()
}

weights

In [ ]:
adaptive_df["adaptive_score"] = (
    weights["semantic"] * adaptive_df["semantic_score_norm"]
    +
    weights["skill"] * adaptive_df["skill_match_score"]
    +
    weights["role"] * adaptive_df["role_match_score"]
    +
    weights["location"] * adaptive_df["location_match_score"]
)

In [ ]:
adaptive_df = (
    adaptive_df
    .sort_values(
        "adaptive_score",
        ascending=False
    )
    .reset_index(drop=True)
)

adaptive_df["adaptive_rank"] = (
    np.arange(len(adaptive_df)) + 1
)

In [ ]:
accepted_after = adaptive_df[
    adaptive_df["feedback"] == "accept"
]

adaptive_rank = accepted_after[
    "adaptive_rank"
].mean()

print("Before:", round(baseline_rank,2))
print("After:", round(adaptive_rank,2))

In [ ]:
# Second adaptive-learning iteration

round2_df = adaptive_df.copy()

# Simulate feedback again on the newly re-ranked list
round2_df["feedback_round2"] = round2_df.apply(simulate_feedback, axis=1)

round2_df["round2_rank_before"] = np.arange(len(round2_df)) + 1

accepted_round2 = round2_df[round2_df["feedback_round2"] == "accept"]
rejected_round2 = round2_df[round2_df["feedback_round2"] == "reject"]

round2_baseline_rank = accepted_round2["round2_rank_before"].mean()

# Compute feature differences between accepted and rejected jobs
skill_diff2 = (
    accepted_round2["skill_match_score"].mean()
    - rejected_round2["skill_match_score"].mean()
)

role_diff2 = (
    accepted_round2["role_match_score"].mean()
    - rejected_round2["role_match_score"].mean()
)

location_diff2 = (
    accepted_round2["location_match_score"].mean()
    - rejected_round2["location_match_score"].mean()
)

# Update weights again
weights["skill"] += 0.50 * skill_diff2
weights["role"] += 0.50 * role_diff2
weights["location"] += 0.50 * location_diff2

# Normalize weights
total = sum(weights.values())
weights = {k: v / total for k, v in weights.items()}

print("Updated weights after Round 2:")
print(weights)

# Recompute adaptive score
round2_df["adaptive_score_round2"] = (
    weights["semantic"] * round2_df["semantic_score_norm"] +
    weights["skill"] * round2_df["skill_match_score"] +
    weights["role"] * round2_df["role_match_score"] +
    weights["location"] * round2_df["location_match_score"]
)

# Re-rank again
round2_df = (
    round2_df
    .sort_values("adaptive_score_round2", ascending=False)
    .reset_index(drop=True)
)

round2_df["round2_rank_after"] = np.arange(len(round2_df)) + 1

accepted_after_round2 = round2_df[round2_df["feedback_round2"] == "accept"]

round2_after_rank = accepted_after_round2["round2_rank_after"].mean()

print("Round 2 Before:", round(round2_baseline_rank, 2))
print("Round 2 After:", round(round2_after_rank, 2))

In [ ]:
adaptive_results = pd.DataFrame([
    {
        "round": 0,
        "avg_accepted_rank": baseline_rank,
        "description": "Initial ranking before feedback"
    },
    {
        "round": 1,
        "avg_accepted_rank": adaptive_rank,
        "description": "After first feedback update"
    },
    {
        "round": 2,
        "avg_accepted_rank": round2_after_rank,
        "description": "After second feedback update"
    }
])

adaptive_results

In [ ]:
print(weights)
print("Round 2 Before:", round2_baseline_rank)
print("Round 2 After:", round2_after_rank)

# Live Job API Ingestion

To satisfy the real-time ingestion requirement, JobPilot integrates
with the Adzuna API to retrieve current job postings.

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import sys

!{sys.executable} -m pip install python-dotenv

In [ ]:
import os
import requests
import pandas as pd
import hashlib
import re
from dotenv import load_dotenv
import os

load_dotenv("file.env")

ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID")
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY")
print("App ID loaded:", ADZUNA_APP_ID is not None)
print("App Key loaded:", ADZUNA_APP_KEY is not None)
def normalize_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def make_dedup_key(title, company, location, description):
    raw = "||".join([
        normalize_text(title),
        normalize_text(company),
        normalize_text(location),
        normalize_text(description)
    ])
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

def fetch_adzuna_jobs(
    query="data analyst",
    country="us",
    results_per_page=50,
    page=1
):
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"

    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query,
        "results_per_page": results_per_page,
        "content-type": "application/json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    return response.json()

def standardize_adzuna_results(api_response):
    rows = []

    for job in api_response.get("results", []):
        title = job.get("title", "")
        company = job.get("company", {}).get("display_name", "")
        location = job.get("location", {}).get("display_name", "")
        description = job.get("description", "")
        job_url = job.get("redirect_url", "")
        date_posted = job.get("created", "")
        salary_min = job.get("salary_min", None)
        salary_max = job.get("salary_max", None)

        rows.append({
            "title": title,
            "company": company,
            "location": location,
            "country": "us",
            "description": description,
            "job_url": job_url,
            "date_posted": date_posted,
            "salary_min": salary_min,
            "salary_max": salary_max,
            "salary_text": "",
            "source": "Adzuna Live API",
            "dedup_key": make_dedup_key(title, company, location, description)
        })

    return pd.DataFrame(rows)

# Fetch live jobs from multiple queries
queries = ["data analyst", "business analyst", "analytics engineer", "data scientist"]

live_dfs = []

for q in queries:
    print(f"Fetching live jobs for: {q}")
    response = fetch_adzuna_jobs(query=q, results_per_page=50)
    live_dfs.append(standardize_adzuna_results(response))

live_jobs_df = pd.concat(live_dfs, ignore_index=True)

before_dedup = len(live_jobs_df)

live_jobs_df = (
    live_jobs_df
    .drop_duplicates(subset=["dedup_key"])
    .reset_index(drop=True)
)

after_dedup = len(live_jobs_df)

print("Live jobs before dedup:", before_dedup)
print("Live jobs after dedup:", after_dedup)

live_jobs_df.head()

In [ ]:
import os

os.makedirs("data", exist_ok=True)

live_jobs_df.to_csv(
    "data/live_jobs_snapshot.csv",
    index=False
)

print("Saved live job snapshot to data/live_jobs_snapshot.csv")